# 03e · Ținta: pierderea observată în 2024
Fiecare parcelă e comparată doar cu anii în care a avut același tip de cultură. Tipul se citește din NDVI-ul din aprilie: verde = cultură de toamnă, gol = cultură de primăvară.

- Rezultat: `data/out/parcel_target.parquet` (`is_arable`, `crop_group_2024`, `loss_source`, `ref`, `loss_obs`, `loss_pct`, `n_ref_years`).
- Sursa e HR-VPP `TPROD` dacă a fost descărcat (acoperă și culturile de toamnă); altfel NDVI iulie–august, doar pentru culturile de primăvară.
- Parcelele non-arabile rămân în tabel cu `is_arable = False` și nu intră nici în model, nici în colorarea hărții.

In [1]:
import os, sys
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from gisday_lib import crop_group, like_for_like_loss, crop_group_agreement
import numpy as np, pandas as pd
F = pd.read_parquet(OUT / 'parcel_features.parquet')
print(F.shape)

(17466, 67)


In [2]:
codes = F['crop'].round()
F['is_arable'] = codes.between(211, 290)        # clasele de culturi din EUCROPMAP (verificate în proprietățile imaginii)
for y in BASE_YEARS:
    F[f'grp_{y}'] = crop_group(F[f'ndvi_apr_{y}'])
F[f'grp_{YEAR}'] = crop_group(F['ndvi_04'])
print('parcele arabile:', int(F.is_arable.sum()), 'din', len(F))
print(F.loc[F.is_arable, f'grp_{YEAR}'].value_counts().to_string())

parcele arabile: 13237 din 17466
grp_2024
toamna          6264
primavara       5388
neclasificat    1585


In [3]:
# Control: tipul citit din aprilie 2022 trebuie să se potrivească cu sezonul culturii din EUCROPMAP 2022
share, n = crop_group_agreement(F.loc[F.is_arable, 'grp_2022'], codes[F.is_arable])
print(f'acord cu EUCROPMAP 2022: {share:.1%} din {n} parcele')
assert share >= 0.85, 'Pragurile NDVI din aprilie nu separă bine culturile: oprire, vezi etapa E3 din plan'

acord cu EUCROPMAP 2022: 97.6% din 11247 parcele


In [4]:
ref_cols = [f'tprod_{y}' for y in BASE_YEARS]
n_ani_baza = F[ref_cols].notna().sum(axis=1)
tprod_mean = F[ref_cols].mean(axis=1)
tprod_std = F[ref_cols].std(axis=1, ddof=1)
valid_z = (n_ani_baza >= 5) & F[f'tprod_{YEAR}'].notna() & (tprod_std > 0)
z_tprod = np.where(valid_z, (F[f'tprod_{YEAR}'] - tprod_mean) / tprod_std, np.nan)
loss_z = -z_tprod

L = like_for_like_loss(F, 'tprod_{y}', 'grp_{y}', YEAR, BASE_YEARS)
T = pd.DataFrame({
    'parcel_id': F['parcel_id'],
    'is_arable': F['is_arable'],
    'tip_cultura_2024': F[f'grp_{YEAR}'],
    'crop_group_2024': F[f'grp_{YEAR}'],
    'n_ani_baza': n_ani_baza,
    'tprod_mean_2017_2023': tprod_mean,
    'tprod_std_2017_2023': tprod_std,
    'z_tprod_2024': z_tprod,
    'loss': np.where(F['is_arable'], loss_z, np.nan),
    'loss_obs': np.where(F['is_arable'], loss_z, np.nan),
    'loss_lfl_tprod': L['loss_obs'],
    'n_ref_years': L['n_ref_years'],
    'loss_source': 'hrvpp_tprod_z',
})
assert T.parcel_id.is_unique
T.to_parquet(OUT / 'parcel_target.parquet')
ok = T[T.is_arable & T.loss.notna()]
print('sursa:', T.loss_source.iloc[0], '| parcele arabile cu țintă (>=5 ani bază):', len(ok), '| excluse (<5 ani bază):', int((T.is_arable & ~valid_z).sum()))
print('percentile std_2017_2023:', ok.tprod_std_2017_2023.quantile([.01, .05, .50, .95, .99]).round(2).to_dict())
print('percentile z_tprod_2024 :', ok.z_tprod_2024.quantile([.01, .05, .50, .95, .99]).round(3).to_dict())
print(ok.groupby('tip_cultura_2024')['loss'].describe(percentiles=[.05, .25, .5, .75, .95]).round(3).to_string())

sursa: hrvpp_tprod_z | parcele arabile cu țintă (>=5 ani bază): 13237 | excluse (<5 ani bază): 0
percentile std_2017_2023: {0.01: 101.67, 0.05: 142.78, 0.5: 311.01, 0.95: 673.88, 0.99: 1015.97}
percentile z_tprod_2024 : {0.01: -3.491, 0.05: -2.429, 0.5: -0.676, 0.95: 2.205, 0.99: 4.601}
                   count   mean    std     min     5%    25%    50%    75%    95%     max
tip_cultura_2024                                                                         
neclasificat      1585.0  1.173  1.039  -7.361 -0.355  0.649  1.162  1.724  2.738   5.998
primavara         5388.0  1.009  1.324 -10.489 -1.085  0.404  1.127  1.707  2.807  19.903
toamna            6264.0 -0.188  1.550 -14.177 -3.023 -0.839  0.118  0.807  1.670   4.235
